# Pairs-trading research notebook

A walk through the StatArb Research Lab workflow using the `statarb` package. Every number is computed from the data loaded below; nothing is hard-coded.

**Research question:** can a pairs strategy based on a statistically justified relationship earn robust out-of-sample returns after costs?

Workflow: load and validate data, split it chronologically, fit on training data only, test for cointegration, build signals, run a two-leg backtest with costs, compare with benchmarks, then check robustness.

By default the notebook uses **labelled synthetic data** so that it runs offline. Set `SOURCE = "yfinance"` to download real prices. The command-line equivalent is `python scripts/run_research.py`.

*Research and education only. Not financial advice.*

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from statarb.config import load_config, resolve_path
from statarb.data_loader import download_yfinance, generate_synthetic_pair
from statarb.data_validation import prepare_pair_from_config
from statarb.evaluation import chronological_split, run_pair_research

pd.set_option("display.width", 140)
config = load_config(ROOT / "config" / "default_config.yaml")
SOURCE = "synthetic"  # "synthetic" (offline) or "yfinance"
TICKERS = ("KO", "PEP")

## 1. Load, validate and align

Rows are only removed, never invented: there is no forward filling. Each asset is validated, then the pair is aligned on common dates.

In [ ]:
if SOURCE == "yfinance":
    cache = resolve_path(config.data.cache_dir)
    asset_a = download_yfinance(TICKERS[0], config.data.start_date, config.data.end_date, cache)
    asset_b = download_yfinance(TICKERS[1], config.data.start_date, config.data.end_date, cache)
else:
    asset_a, asset_b = generate_synthetic_pair(n_obs=2520, seed=42, cointegrated=True)

pair = prepare_pair_from_config(asset_a, asset_b, config.data)
print("synthetic:", pair.is_synthetic)
pd.DataFrame({pair.ticker_a: pair.validation_a.summary(), pair.ticker_b: pair.validation_b.summary()})

## 2. Chronological splits

Training (60%) is used to fit everything. Validation (20%) may be used to compare thresholds. The final test (20%) stays untouched until the specification is fixed.

In [ ]:
splits = chronological_split(pair.prices.index, config.splits.train, config.splits.validation, config.splits.test)
pd.DataFrame([{"period": k, "start": v.start_date, "end": v.end_date, "n": v.n_obs} for k, v in splits.items()])

## 3. Training-only estimation and the Engle-Granger test

The test period stays locked in this first run (`reveal_test=False`).

In [ ]:
result = run_pair_research(pair, config, reveal_test=False)
c = result.model.coint
print(c.decision)
print(c.interpretation())
pd.Series(result.model.summary())

## 4. In-sample vs validation performance

The training-period backtest is in-sample, so it is not evidence of performance. The validation period is out-of-sample.

In [ ]:
keys = ["cumulative_return", "gross_cumulative_return", "sharpe_ratio", "max_drawdown", "completed_trades",
        "win_rate", "pct_time_invested", "total_transaction_costs"]
pd.DataFrame({k: {m: p.metrics[m] for m in keys} for k, p in result.periods.items()})

In [ ]:
result.periods["validation"].comparison

## 5. Robustness: validation grid, costs and walk-forward

The grid is evaluated on validation data only. Choosing its best cell overstates expected performance, because many combinations were tried (multiple testing).

In [ ]:
print("grid points evaluated:", len(result.validation_grid))
result.validation_grid.pivot(index="exit_z", columns="entry_z", values="sharpe_ratio").round(2)

In [ ]:
result.cost_sensitivity["validation"]

In [ ]:
wf = result.walk_forward
wf.windows[["window", "fit_start", "eval_start", "eval_end", "hedge_ratio", "eg_p_value", "traded", "completed_trades", "window_return"]].round(4)

## 6. Final test (run once, after the specification is fixed)

The thresholds come from the config (entry 2.0, exit 0.5). They were fixed in advance, not tuned on the test period.

In [ ]:
final = run_pair_research(pair, config, reveal_test=True)
final.periods["test"].comparison

## 7. Interpretation checklist

- Did the pair pass the Engle-Granger test **on training data**? A pass is evidence, not proof.
- Is the validation and test performance positive **after costs**? How sensitive is it to the cost assumption?
- Is the hedge ratio stable across walk-forward windows? Instability suggests a structural break.
- How many pairs and parameters were tried before reaching this result? Every extra trial weakens the evidence.
- Is short selling actually feasible for both legs, and at what borrow cost?

## 8. Machine-learning layer: traditional vs ML-only vs hybrid

Target: `y_t = 1` if `|z_{t+H}| <= |z_t| - tau` (H = 5, tau = 0.25), on rows with `|z_t| >= 1`. Features use data up to `t` only. Labels are purged at every period boundary. Hyperparameters are tuned on validation, never on test. Strategies A (traditional), B (ML-only) and C (hybrid) share dates, execution lag, sizing and costs.

In [ ]:
from statarb.ml_evaluation import classification_summary, run_ml_research

ml = run_ml_research(final)  # 'final' is the revealed-test research result from section 6
ml.sample_table[["start", "end", "rows_used", "positive_rate"]]

In [ ]:
keys = ["n_samples", "positive_rate", "majority_class_accuracy", "accuracy", "f1", "roc_auc", "brier_score"]
classification_summary(ml).loc[keys].astype(float).round(3)

In [ ]:
rows = ["cumulative_return", "gross_cumulative_return", "sharpe_ratio", "max_drawdown", "completed_trades", "total_transaction_costs"]
ml.comparisons["test"].table.loc[rows].round(4)

In [ ]:
ml.walk_forward.table.loc[rows].round(4)

Accuracy should be read against the majority-class rate, and ROC-AUC against 0.5. A variant only deserves attention if it beats A on validation, test *and* walk-forward after costs, with enough trades. Otherwise the difference is indistinguishable from chance.

## 9. The audited v2 protocol (pre-registered)

Sections 1-8 use the original (baseline) settings. The Phase 1 audit (`docs/audit/PHASE1_AUDIT.md`) found a hyperparameter look-ahead in the ML walk-forward, miscalibrated probabilities, special NSE sessions traded as normal bars, and a test period that had been viewed repeatedly. The v2 protocol (`config/research_v2.yaml`, `docs/research/PREREGISTRATION.md`) fixes these. It is run with `scripts/run_experiment.py` in three stages: development (2015-2024), freeze, and a one-time holdout (2025-01-01 to 2026-09-30). The cell below summarises the saved runs, if present. The full analysis is in `docs/research/RESEARCH_REPORT.md`.

In [ ]:
import json

runs = sorted((ROOT / "reports" / "experiments").glob("*_*")) if (ROOT / "reports" / "experiments").exists() else []
if not runs:
    print("No v2 experiment runs found. Run: python scripts/run_experiment.py --stage development")
for run in runs:
    summary = json.loads((run / "summary.json").read_text())
    sharpe = {k: round(v["point"], 3) for k, v in summary["bootstrap"]["sharpe"].items()}
    auc = {m: round(c.get("roc_auc") or float("nan"), 3) for m, c in summary["classification"].items()}
    print(run.name, summary["evaluation_dates"], "pooled Sharpe:", sharpe, "ROC-AUC:", auc)
    if (run / "decision.json").exists():
        print("   pre-registered decision:", json.loads((run / "decision.json").read_text())["overall"])